# The association of polypharmacy with sociodemographic and clinical outcomes

## Notes before running the scripts
- Remember to replace all data and working directory paths with your actual paths.
- Dependency: haven, tidyverse, broom, tidyr, glue, writexl

In [ ]:
library(haven)
library(tidyverse)
library(broom)
library(writexl)
library(tidyr)
library(glue)


In [ ]:
logistic_regression_adjusted_sociodemographic <- function(data, predictors, outcomes, 
                                                            ref_predictor = NULL, 
                                                            level_order = NULL, 
                                                            ref_outcome = "0", 
                                                            excel_file = "logistic_results_formatted.xlsx") {
  
  # Check if outcomes exist
  missing_outcomes <- outcomes[!outcomes %in% names(data)]
  if(length(missing_outcomes) > 0) stop(paste("Missing outcomes:", paste(missing_outcomes, collapse = ", ")))

  # Check if predictors exist
  missing_predictors <- predictors[!predictors %in% names(data)]
  if(length(missing_predictors) > 0) stop(paste("Missing predictors:", paste(missing_predictors, collapse = ", ")))

  if(length(outcomes) != 2) stop("Exactly two outcome variables must be provided")

  # Prepare factors
  for (pred in predictors) {
    if (!is.factor(data[[pred]]) && !is.character(data[[pred]])) {
      warning(paste("Converting", pred, "to factor"))
      data[[pred]] <- as.factor(data[[pred]])
    }

    if (!is.null(level_order) && pred %in% names(level_order)) {
      data[[pred]] <- factor(data[[pred]], levels = level_order[[pred]])
    }

    if (!is.null(ref_predictor) && pred %in% names(ref_predictor)) {
      data[[pred]] <- relevel(data[[pred]], ref = ref_predictor[[pred]])
    }
  }

  for (outcome in outcomes) {
    if (is.numeric(data[[outcome]])) {
      if (!all(data[[outcome]] %in% c(0, 1))) stop(paste("Outcome", outcome, "must be binary (0/1)"))
      data[[outcome]] <- factor(data[[outcome]], levels = c("0", "1"))
    }
    data[[outcome]] <- relevel(data[[outcome]], ref = ref_outcome)
  }

  results_list <- list()

  for (outcome in outcomes) {

    formula <- as.formula(paste(outcome, "~", paste(c(predictors,c("log_exposure_std")), collapse = " + ")))
    model <- glm(formula, data = data, family = "binomial")
    
    coef_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE)
    coef_table$Outcome <- outcome
          
    results_list[[outcome]] <- coef_table
  }

  combined <- do.call(rbind, results_list)
  combined <- combined[combined$term != "(Intercept)", ]
  
  # Round and format
  combined <- combined %>%
    mutate(
      Odds_Ratio = round(estimate, 2),
      CI_Lower = round(conf.low, 2),
      CI_Upper = round(conf.high, 2),
      P_Value = round(p.value, 3),
      OR_CI = sprintf("%.2f [%.2f, %.2f]", Odds_Ratio, CI_Lower, CI_Upper),
      P_Value_Formatted = ifelse(P_Value < 0.001, "<0.001", sprintf("%.3f", P_Value))
    )

  # Format final table
  final_table <- data.frame(
    Characteristics = character(),
    `Lifetime psychotropic OR (95% CI)` = character(),
    `P value` = character(),
    `Concurrent psychotropic OR (95% CI)` = character(),
    `P value` = character(),
    stringsAsFactors = FALSE
  )

  for (pred in predictors) {
    levels_list <- levels(data[[pred]])
    ref <- ref_predictor[[pred]]
    
    pred_rows <- list()
    pred_rows[[1]] <- data.frame(
      Characteristics = ref,
      `Lifetime psychotropic OR (95% CI)` = "1 [Reference]",
      `P value` = "NA",
      `Concurrent psychotropic OR (95% CI)` = "1 [Reference]",
      `P value` = "NA",
      stringsAsFactors = FALSE
    )

    for (lvl in levels_list) {
      if (lvl == ref) next
      term_name <- paste0(pred, lvl)
      cum_row <- combined[combined$Outcome == outcomes[1] & combined$term == term_name, ]
      con_row <- combined[combined$Outcome == outcomes[2] & combined$term == term_name, ]

      pred_rows[[length(pred_rows) + 1]] <- data.frame(
        Characteristics = lvl,
        `Lifetime psychotropic OR (95% CI)` = if(nrow(cum_row) > 0) cum_row$OR_CI else "NA",
        `P value` = if(nrow(cum_row) > 0) cum_row$P_Value_Formatted else "NA",
        `Concurrent psychotropic OR (95% CI)` = if(nrow(con_row) > 0) con_row$OR_CI else "NA",
        `P value` = if(nrow(con_row) > 0) con_row$P_Value_Formatted else "NA",
        stringsAsFactors = FALSE
      )
    }

    pred_table <- do.call(rbind, pred_rows)
    pred_table <- rbind(
      data.frame(
        Characteristics = pred,
        `Lifetime psychotropic OR (95% CI)` = "",
        `P value` = "",
        `Concurrent psychotropic OR (95% CI)` = "",
        `P value` = "",
        stringsAsFactors = FALSE
      ),
      pred_table
    )
    final_table <- rbind(final_table, pred_table)
  }

  # Output
  cat("\nFormatted Logistic Regression Results (All Predictors Simultaneously)\n")
  cat("----------------------------------------------------------------------\n")
  print(final_table)
  write_xlsx(final_table, path = excel_file)
  cat("\nResults saved to", excel_file, "\n")
  
  invisible(final_table)
}

In [ ]:
all_polypharmacy_outcome_data <- read.csv("/working_directory/cohort_polypharmacy_with_outcome.csv")


In [ ]:
# Define predictors and outcomes
predictors <- c("sex_at_birth", "age_group", "race", "education_level", "marital_status", "income_level")
outcomes <- c("lifetime_polypharmacy_binary", "concurrent_polypharmacy_binary")

# Specify reference levels for predictors (match the table)
ref_predictor <- list(
  sex_at_birth = "Male",
  age_group = "18-29",
  race = "Black or African American",
  education_level = "Advanced degree",
  marital_status = "Married",
  income_level = '>200k'
)


level_order <- list(
  sex_at_birth = c( "Male", "Female","Other"),
  age_group = c("18-29", "30-39", "40-49", "50-59", "60-69", ">=70"),
  race = c('White','Black or African American','Another single population',
    'More than one population',
    'Other'),
  education_level = c("Advanced degree",
                        "College graduate", 
                        "College one to three", 
                        "Twelve or GED", 
                        "Nine through eleven", 
                        "One to eight or Never attended", 
                        "Other"),
  marital_status = c('Married', "Living with partner", 'Separated', 'Divorced', 'Widowed', 'Never married', 'Other'),
  income_level = c('>200k','150k-200k','100k-150k','75k-100k','50k-75k','35k-50k', '25k-35k','10k-25k','<10k','Other')
)

output_f <- "/working_directory/Table_4_adjusted_associations_of_polypharmacy_with_sociodemographic.xlsx"

logistic_regression_adjusted_sociodemographic(all_polypharmacy_outcome_data, predictors, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,
                                           ref_outcome = "0", 
                                           excel_file = output_f)

In [ ]:
# Required packages
library(writexl)  # For writing to Excel
library(broom)    # For tidy model output
library(tidyr)    # For reshaping data

logistic_regression_unadjusted <- function(data, predictors, outcomes, 
                                               ref_predictor = NULL, 
                                               level_order = NULL, 
                                               ref_outcome = "0", 
                                               excel_file = "logistic_results_formatted.xlsx") {
  # Function to perform logistic regression for multiple predictors and two outcomes, 
  # with specified order of predictors and levels, and save to Excel
  
  # Check if outcomes exist
  missing_outcomes <- outcomes[!outcomes %in% names(data)]
  if(length(missing_outcomes) > 0) {
    stop(paste("Outcome(s) not found in dataset:", paste(missing_outcomes, collapse = ", ")))
  }
  
  # Check if all predictors exist
  missing_predictors <- predictors[!predictors %in% names(data)]
  if(length(missing_predictors) > 0) {
    stop(paste("Predictor(s) not found in dataset:", paste(missing_predictors, collapse = ", ")))
  }
  
  # Validate exactly two outcomes
  if(length(outcomes) != 2) {
    stop("Exactly two outcome variables must be provided")
  }
  
  # Convert predictors to factors and set level order if specified
  for(pred in predictors) {
    # Convert to factor if not already
    if(!is.factor(data[[pred]]) && !is.character(data[[pred]])) {
      warning(paste("Predictor", pred, "is not categorical. Converting to factor."))
      data[[pred]] <- as.factor(data[[pred]])
    }
    
    # Set level order if specified in level_order
    if(!is.null(level_order) && pred %in% names(level_order)) {
      if(!all(level_order[[pred]] %in% levels(as.factor(data[[pred]])))) {
        stop(paste("Specified levels for", pred, "do not match data levels"))
      }
      data[[pred]] <- factor(data[[pred]], levels = level_order[[pred]])
    }
    
    # Set predictor reference level if specified
    if(!is.null(ref_predictor) && pred %in% names(ref_predictor)) {
      if(!ref_predictor[[pred]] %in% levels(data[[pred]])) {
        stop(paste("Specified reference level for", pred, "not found in data"))
      }
      data[[pred]] <- relevel(data[[pred]], ref = ref_predictor[[pred]])
    }
  }
  
  # Convert outcomes to factors
  for(outcome in outcomes) {
    if(is.numeric(data[[outcome]])) {
      if(!all(data[[outcome]] %in% c(0, 1))) {
        stop(paste("Outcome variable", outcome, "must contain only 0 and 1"))
      }
      data[[outcome]] <- factor(data[[outcome]], levels = c("0", "1"))
    }
    
    # Set outcome reference level
    if(!is.null(ref_outcome)) {
      if(!ref_outcome %in% c("0", "1")) {
        stop("Reference level for outcome must be '0' or '1'")
      }
      data[[outcome]] <- relevel(data[[outcome]], ref = ref_outcome)
    } else {
      data[[outcome]] <- relevel(data[[outcome]], ref = "0")  # Default to 0
    }
  }
  
  # Initialize results list
  results_list <- list()
  
  # Loop through each outcome
  for(outcome in outcomes) {
    for(pred in predictors) {
      # Fit logistic regression model
      formula <- as.formula(paste(outcome, "~", pred))
      model <- glm(formula, data = data, family = "binomial")
      #model <- logistf(formula, data = data)
      # Extract coefficients, OR, CI, and p-values
      coef_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE)
      coef_table <- coef_table[, c("term", "estimate", "conf.low", "conf.high", "p.value")]
      names(coef_table) <- c("Term", "Odds_Ratio", "CI_Lower", "CI_Upper", "P_Value")
      
      # Add predictor and outcome to results
      coef_table$Predictor <- pred
      coef_table$Outcome <- outcome
      results_list[[paste(outcome, pred, sep = "_")]] <- coef_table
    }
  }
  
  # Combine all results into one data frame
  results_df <- do.call(rbind, results_list)
  
  # Round numeric columns
  results_df$Odds_Ratio <- round(results_df$Odds_Ratio, 2)
  results_df$CI_Lower <- round(results_df$CI_Lower, 2)
  results_df$CI_Upper <- round(results_df$CI_Upper, 2)
  results_df$P_Value <- round(results_df$P_Value, 3)
  
  # Format OR (95% CI) column
  results_df$OR_CI <- sprintf("%.2f [%.2f, %.2f]", results_df$Odds_Ratio, results_df$CI_Lower, results_df$CI_Upper)
  
  # Format p-value column
  results_df$P_Value_Formatted <- ifelse(results_df$P_Value < 0.001, "<0.001", sprintf("%.3f", results_df$P_Value))
  
  # Create a formatted table for each predictor
  formatted_results <- list()
  
  for(pred in predictors) {
    pred_data <- results_df[results_df$Predictor == pred, ]
    
    # Create a table for this predictor with exactly 5 columns
    pred_table <- data.frame(
      Characteristics = character(),
      Cumulative_OR_CI = character(),
      Cumulative_P = character(),
      Concurrent_OR_CI = character(),
      Concurrent_P = character(),
      stringsAsFactors = FALSE
    )
    
    # Get levels for this predictor
    pred_levels <- levels(data[[pred]])
    
    # Debug: Check pred_levels
    if(length(pred_levels) == 0) {
      stop(paste("No levels found for predictor", pred, "- check data and level_order"))
    }
    
    # First row: Reference level
    pred_table[1, ] <- c(pred_levels[1], "1 [Reference]", "NA", "1 [Reference]", "NA")
    
    # Add other levels
    for(i in 2:length(pred_levels)) {
      level <- pred_levels[i]
      term <- paste0(pred, level)
      
      # Cumulative outcome (first outcome)
      cum_row <- pred_data[pred_data$Outcome == outcomes[1] & pred_data$Term == term, ]
      cum_or_ci <- if(nrow(cum_row) > 0) cum_row$OR_CI else "NA"
      cum_p <- if(nrow(cum_row) > 0) cum_row$P_Value_Formatted else "NA"
      
      # Concurrent outcome (second outcome)
      con_row <- pred_data[pred_data$Outcome == outcomes[2] & pred_data$Term == term, ]
      con_or_ci <- if(nrow(con_row) > 0) con_row$OR_CI else "NA"
      con_p <- if(nrow(con_row) > 0) con_row$P_Value_Formatted else "NA"
      
      # Add row to pred_table
      pred_table[i, ] <- c(level, cum_or_ci, cum_p, con_or_ci, con_p)
    }
    
    # Add predictor name as a header row
    pred_table <- rbind(
      data.frame(
        Characteristics = pred,
        Cumulative_OR_CI = "",
        Cumulative_P = "",
        Concurrent_OR_CI = "",
        Concurrent_P = "",
        stringsAsFactors = FALSE
      ),
      pred_table
    )
    
    formatted_results[[pred]] <- pred_table
  }
  
  # Combine all predictor tables in the specified order
  final_table <- do.call(rbind, formatted_results[predictors])
  
  # Rename columns to match the desired format
  names(final_table) <- c("Characteristics", 
                         "Lifetime psychotropic OR (95% CI)", 
                         "P value", 
                         "Concurrent psychotropic OR (95% CI)", 
                         "P value")
  # Print results to console
  cat("\nFormatted Logistic Regression Results\n")
  cat("-------------------------------------------------\n")
  print(final_table)
  
  # Save to Excel
  write_xlsx(final_table, path = excel_file)
  cat("\nResults saved to", excel_file, "\n")
  
  # Return results invisibly
  invisible(final_table)
}

In [ ]:
# Define predictors and outcomes
predictors <- c("sex_at_birth", "age_group", "race", "education_level", "marital_status", "income_level")
outcomes <- c("lifetime_polypharmacy_binary", "concurrent_polypharmacy_binary")

# Specify reference levels for predictors (match the table)
ref_predictor <- list(
  sex_at_birth = "Male",
  age_group = "18-29",
  race = "Black or African American",
  education_level = "Advanced degree",
  marital_status = "Married",
  income_level = '>200k'
)


level_order <- list(
  sex_at_birth = c( "Male", "Female","Other"),
  age_group = c("18-29", "30-39", "40-49", "50-59", "60-69", ">=70"),
  race = c('White','Black or African American','Another single population',
    'More than one population',
    'Other'),
  education_level = c("Advanced degree",
                        "College graduate", 
                        "College one to three", 
                        "Twelve or GED", 
                        "Nine through eleven", 
                        "One to eight or Never attended", 
                        "Other"),
  marital_status = c('Married', "Living with partner", 'Separated', 'Divorced', 'Widowed', 'Never married', 'Other'),
  income_level = c('>200k','150k-200k','100k-150k','75k-100k','50k-75k','35k-50k', '25k-35k','10k-25k','<10k','Other')
)

output_f <- "/working_directory/Table_4_unadjusted_associations_of_polypharmacy_with_sociodemographic.xlsx"
# Run analysis and save to Excel
logistic_regression_unadjusted(all_polypharmacy_outcome_data, predictors, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,
                                           ref_outcome = "0", 
                                           excel_file = output_f)

# polypharmacy ~ CCI + ERV + IV

In [ ]:
logistic_regression_adjusted <- function(data, predictors,convariates, outcomes, 
                                                            ref_predictor = NULL, 
                                                            level_order = NULL, 
                                                            ref_outcome = "0", 
                                                            excel_file = "logistic_results_formatted.xlsx") {
  
  # Check if outcomes exist
  missing_outcomes <- outcomes[!outcomes %in% names(data)]
  if(length(missing_outcomes) > 0) stop(paste("Missing outcomes:", paste(missing_outcomes, collapse = ", ")))

  # Check if predictors exist
  missing_predictors <- predictors[!predictors %in% names(data)]
  if(length(missing_predictors) > 0) stop(paste("Missing predictors:", paste(missing_predictors, collapse = ", ")))

  if(length(outcomes) != 2) stop("Exactly two outcome variables must be provided")

  # Prepare factors
  for (pred in predictors) {
    if (!is.factor(data[[pred]]) && !is.character(data[[pred]])) {
      warning(paste("Converting", pred, "to factor"))
      data[[pred]] <- as.factor(data[[pred]])
    }

    if (!is.null(level_order) && pred %in% names(level_order)) {
      data[[pred]] <- factor(data[[pred]], levels = level_order[[pred]])
    }

    if (!is.null(ref_predictor) && pred %in% names(ref_predictor)) {
      data[[pred]] <- relevel(data[[pred]], ref = ref_predictor[[pred]])
    }
  }

  for (outcome in outcomes) {
    if (is.numeric(data[[outcome]])) {
      if (!all(data[[outcome]] %in% c(0, 1))) stop(paste("Outcome", outcome, "must be binary (0/1)"))
      data[[outcome]] <- factor(data[[outcome]], levels = c("0", "1"))
    }
    data[[outcome]] <- relevel(data[[outcome]], ref = ref_outcome)
  }

  results_list <- list()

  for (outcome in outcomes) {
    pred_cov <- c(predictors, convariates)
    formula <- as.formula(paste(outcome, "~", paste(pred_cov, collapse = " + ")))
    model <- glm(formula, data = data, family = "binomial")
    coef_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE)
    coef_table$Outcome <- outcome
    results_list[[outcome]] <- coef_table
  }

  combined <- do.call(rbind, results_list)
  combined <- combined[combined$term != "(Intercept)", ]
  
  # Round and format
  combined <- combined %>%
    mutate(
      Odds_Ratio = round(estimate, 2),
      CI_Lower = round(conf.low, 2),
      CI_Upper = round(conf.high, 2),
      P_Value = round(p.value, 3),
      OR_CI = sprintf("%.2f [%.2f, %.2f]", Odds_Ratio, CI_Lower, CI_Upper),
      P_Value_Formatted = ifelse(P_Value < 0.001, "<0.001", sprintf("%.3f", P_Value))
    )

  # Format final table
  final_table <- data.frame(
    Characteristics = character(),
    `Lifetime psychotropic OR (95% CI)` = character(),
    `P value` = character(),
    `Concurrent psychotropic OR (95% CI)` = character(),
    `P value` = character(),
    stringsAsFactors = FALSE
  )

  for (pred in predictors) {
    levels_list <- levels(data[[pred]])
    ref <- ref_predictor[[pred]]
    
    pred_rows <- list()
    pred_rows[[1]] <- data.frame(
      Characteristics = ref,
      `Lifetime psychotropic OR (95% CI)` = "1 [Reference]",
      `P value` = "NA",
      `Concurrent psychotropic OR (95% CI)` = "1 [Reference]",
      `P value` = "NA",
      stringsAsFactors = FALSE
    )

    for (lvl in levels_list) {
      if (lvl == ref) next
      term_name <- paste0(pred, lvl)
      cum_row <- combined[combined$Outcome == outcomes[1] & combined$term == term_name, ]
      con_row <- combined[combined$Outcome == outcomes[2] & combined$term == term_name, ]

      pred_rows[[length(pred_rows) + 1]] <- data.frame(
        Characteristics = lvl,
        `Lifetime psychotropic OR (95% CI)` = if(nrow(cum_row) > 0) cum_row$OR_CI else "NA",
        `P value` = if(nrow(cum_row) > 0) cum_row$P_Value_Formatted else "NA",
        `Concurrent psychotropic OR (95% CI)` = if(nrow(con_row) > 0) con_row$OR_CI else "NA",
        `P value` = if(nrow(con_row) > 0) con_row$P_Value_Formatted else "NA",
        stringsAsFactors = FALSE
      )
    }

    pred_table <- do.call(rbind, pred_rows)
    pred_table <- rbind(
      data.frame(
        Characteristics = pred,
        `Lifetime psychotropic OR (95% CI)` = "",
        `P value` = "",
        `Concurrent psychotropic OR (95% CI)` = "",
        `P value` = "",
        stringsAsFactors = FALSE
      ),
      pred_table
    )
    final_table <- rbind(final_table, pred_table)
  }

  # Output
  cat("\nFormatted Logistic Regression Results (All Predictors Simultaneously)\n")
  cat("----------------------------------------------------------------------\n")
  print(final_table)
  write_xlsx(final_table, path = excel_file)
  cat("\nResults saved to", excel_file, "\n")
  
  invisible(final_table)
}

In [ ]:
convariates <- c("age", "sex", 'race_group', 'education_level', 'marital_status', 'income_level', 'log_exposure_std')

predictors <- c("CCI_group", "ERV_count_group", "IV_count_group")#, "death_group"
outcomes <- c("lifetime_polypharmacy_binary", "concurrent_polypharmacy_binary")

# Specify reference levels for predictors (match the table)
ref_predictor <- list(
  CCI_group = "0",
  ERV_count_group = "0",
  IV_count_group = "0"
)

# Specify the order of levels for each predictor (match the table)
level_order <- list(
  #CCI_group = c( "0", "1","2-5", "6-9", ">=10"),
  CCI_group = c( "0", "1-2","3-4", ">=5"),
  ERV_count_group = c("0", "1-5", "6-10", "11-20", ">=21"),
  IV_count_group = c("0", "1-5", "6-10", "11-20", ">=21")    
  #ERV_count_group = c("0", "1-5", "6-10", "11-20", "21-30", "31-40", ">=41"),
  #IV_count_group = c("0", "1-5", "6-10", "11-20", "21-30", "31-40", ">=41")
)

output_f <- "/working_directory/Table_5_adjusted_associations_of_polypharmacy_with_cci_ERIV.xlsx"
# Run analysis and save to Excel
logistic_regression_adjusted(all_polypharmacy_outcome_data, predictors, convariates, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,
                                           ref_outcome = "0", 
                                           excel_file = output_f)

In [ ]:
predictors <- c("CCI_group", "ERV_count_group", "IV_count_group")#, "death_group"
outcomes <- c("lifetime_polypharmacy_binary", "concurrent_polypharmacy_binary")

# Specify reference levels for predictors (match the table)
ref_predictor <- list(
  CCI_group = "0",
  ERV_count_group = "0",
  IV_count_group = "0"
)

# Specify the order of levels for each predictor (match the table)
level_order <- list(
  CCI_group = c( "0", "1-2","3-4", ">=5"),
  ERV_count_group = c("0", "1-5", "6-10", "11-20", ">=21"),
  IV_count_group = c("0", "1-5", "6-10", "11-20", ">=21")  
)

output_f <- "/working_directory/Table_5_unadjusted_associations_of_polypharmacy_with_cci_ERIV.xlsx"
# Run analysis and save to Excel
logistic_regression_unadjusted(all_polypharmacy_outcome_data, predictors, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,
                                           ref_outcome = "0", 
                                           excel_file = output_f)

In [ ]:
# Required packages
library(writexl)  # For writing to Excel
library(broom)    # For tidy model output
library(tidyr)    # For reshaping data

poisson_regression_unadjusted <- function(data, predictors, outcomes, 
                                               ref_predictor = NULL, 
                                               level_order = NULL, 
                                               excel_file = "logistic_results_formatted.xlsx") {

  # Check if outcomes exist
  missing_outcomes <- outcomes[!outcomes %in% names(data)]
  if(length(missing_outcomes) > 0) {
    stop(paste("Outcome(s) not found in dataset:", paste(missing_outcomes, collapse = ", ")))
  }
  
  # Check if all predictors exist
  missing_predictors <- predictors[!predictors %in% names(data)]
  if(length(missing_predictors) > 0) {
    stop(paste("Predictor(s) not found in dataset:", paste(missing_predictors, collapse = ", ")))
  }
  
  # Convert predictors to factors and set level order if specified
  for(pred in predictors) {
    if(!is.factor(data[[pred]]) && !is.character(data[[pred]])) {
      warning(paste("Predictor", pred, "is not categorical. Converting to factor."))
      data[[pred]] <- as.factor(data[[pred]])
    }
    
    if(!is.null(level_order) && pred %in% names(level_order)) {
      if(!all(level_order[[pred]] %in% levels(as.factor(data[[pred]])))) {
        stop(paste("Specified levels for", pred, "do not match data levels"))
      }
      data[[pred]] <- factor(data[[pred]], levels = level_order[[pred]])
    }
    
    # Set predictor reference level if specified
    if(!is.null(ref_predictor) && pred %in% names(ref_predictor)) {
      if(!ref_predictor[[pred]] %in% levels(data[[pred]])) {
        stop(paste("Specified reference level for", pred, "not found in data"))
      }
      data[[pred]] <- relevel(data[[pred]], ref = ref_predictor[[pred]])
    }
  }
  
  # Initialize results list
  results_list <- list()
  
  # Loop through each outcome
  for(outcome in outcomes) {
    for(pred in predictors) {
      formula <- as.formula(paste(outcome, "~", pred))
      model <- glm(formula, family = "poisson", data = data)

      coef_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE)
      coef_table <- coef_table[, c("term", "estimate", "conf.low", "conf.high", "p.value")]
      names(coef_table) <- c("Term", "Odds_Ratio", "CI_Lower", "CI_Upper", "P_Value")
      
      # Add predictor and outcome to results
      coef_table$Predictor <- pred
      coef_table$Outcome <- outcome
      results_list[[paste(outcome, pred, sep = "_")]] <- coef_table
    }
  }
  
  # Combine all results into one data frame
  results_df <- do.call(rbind, results_list)
  
  # Round numeric columns
  results_df$Odds_Ratio <- round(results_df$Odds_Ratio, 2)
  results_df$CI_Lower <- round(results_df$CI_Lower, 2)
  results_df$CI_Upper <- round(results_df$CI_Upper, 2)
  results_df$P_Value <- round(results_df$P_Value, 3)
  
  # Format OR (95% CI) column
  results_df$OR_CI <- sprintf("%.2f [%.2f, %.2f]", results_df$Odds_Ratio, results_df$CI_Lower, results_df$CI_Upper)
  
  # Format p-value column
  results_df$P_Value_Formatted <- ifelse(results_df$P_Value < 0.001, "<0.001", sprintf("%.3f", results_df$P_Value))
  print(results_df)
  formatted_results <- list()
  
  for(pred in predictors) {
    pred_data <- results_df[results_df$Predictor == pred, ]
    
    pred_table <- data.frame(
      Characteristics = character(),
      Concurrent_OR_CI = character(),
      Concurrent_P = character(),
      stringsAsFactors = FALSE
    )
    
    pred_levels <- levels(data[[pred]])
    
    if(length(pred_levels) == 0) {
      stop(paste("No levels found for predictor", pred, "- check data and level_order"))
    }
    
    pred_table[1, ] <- c(pred_levels[1], "1 [Reference]", "NA")
    
    for(i in 2:length(pred_levels)) {
      level <- pred_levels[i]
      term <- paste0(pred, level)
      
      con_row <- pred_data[pred_data$Outcome == outcomes[1] & pred_data$Term == term, ]
      con_or_ci <- if(nrow(con_row) > 0) con_row$OR_CI else "NA"
      con_p <- if(nrow(con_row) > 0) con_row$P_Value_Formatted else "NA"
      
      pred_table[i, ] <- c(level, con_or_ci, con_p)
    }
    
    pred_table <- rbind(
      data.frame(
        Characteristics = pred,
        Concurrent_OR_CI = "",
        Concurrent_P = "",
        stringsAsFactors = FALSE
      ),
      pred_table
    )
    formatted_results[[pred]] <- pred_table
  }
  
  # Combine all predictor tables in the specified order
  final_table <- do.call(rbind, formatted_results[predictors])
  
  # Rename columns to match the desired format
  names(final_table) <- c("Characteristics", 
                         "Concurrent psychotropic polypharmacy OR (95% CI)", 
                         "P value")
  # Print results to console
  cat("\nFormatted Logistic Regression Results\n")
  cat("-------------------------------------------------\n")
  print(final_table)
  
  # Save to Excel
  write_xlsx(final_table, path = excel_file)
  cat("\nResults saved to", excel_file, "\n")
  
  # Return results invisibly
  invisible(final_table)
}


In [ ]:
predictors <- c("CCI_group", "ERV_count_group", "IV_count_group")#, "death_group"
outcomes <- c("concurrent_polypharmacy")# "lifetime_polypharmacy",

#Specify reference levels for predictors (match the table)
ref_predictor <- list(
  CCI_group = "0",
  ERV_count_group = "0",
  IV_count_group = "0"
)

#Specify the order of levels for each predictor (match the table)
level_order <- list(
#   CCI_group = c( "0", "1","2-5", "6-9", ">=10"),
    CCI_group = c( "0", "1-2","3-4", ">=5"),
  ERV_count_group = c("0", "1-5", "6-10", "11-20", ">=21"),
  IV_count_group = c("0", "1-5", "6-10", "11-20", ">=21")  
)

output_f <- "/working_directory/Table_6_unadjusted_associations_of_polypharmacy_with_cci_ERIV.xlsx"

poisson_regression_unadjusted(all_polypharmacy_outcome_data, predictors, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,                         
                                           excel_file = output_f)

In [ ]:
poisson_regression_adjusted <- function(data, predictors,convariates, outcomes, 
                                                            ref_predictor, 
                                                            level_order, 
                                                            excel_file = "results_formatted.xlsx") {
  
  # Check if outcomes exist
  missing_outcomes <- outcomes[!outcomes %in% names(data)]
  if(length(missing_outcomes) > 0) stop(paste("Missing outcomes:", paste(missing_outcomes, collapse = ", ")))

  # Check if predictors exist
  missing_predictors <- predictors[!predictors %in% names(data)]
  if(length(missing_predictors) > 0) stop(paste("Missing predictors:", paste(missing_predictors, collapse = ", ")))

  # Prepare factors
  for (pred in predictors) {
    if (!is.factor(data[[pred]]) && !is.character(data[[pred]])) {
      warning(paste("Converting", pred, "to factor"))
      data[[pred]] <- as.factor(data[[pred]])
    }

    if (!is.null(level_order) && pred %in% names(level_order)) {
      data[[pred]] <- factor(data[[pred]], levels = level_order[[pred]])
    }

    if (!is.null(ref_predictor) && pred %in% names(ref_predictor)) {
      data[[pred]] <- relevel(data[[pred]], ref = ref_predictor[[pred]])
    }
  }


  results_list <- list()
      
  for (outcome in outcomes) {
    pred_cov <- c(predictors, convariates)
    formula <- as.formula(paste(outcome, "~", paste(pred_cov, collapse = " + ")))
    model <- glm(formula, family = "poisson", data = data)
    coef_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE)
    coef_table$Outcome <- outcome
    results_list[[outcome]] <- coef_table
  }

  combined <- do.call(rbind, results_list)
  combined <- combined[combined$term != "(Intercept)", ]
  
  # Round and format
  combined <- combined %>%
    mutate(
      Odds_Ratio = round(estimate, 2),
      CI_Lower = round(conf.low, 2),
      CI_Upper = round(conf.high, 2),
      P_Value = round(p.value, 3),
      OR_CI = sprintf("%.2f [%.2f, %.2f]", Odds_Ratio, CI_Lower, CI_Upper),
      P_Value_Formatted = ifelse(P_Value < 0.001, "<0.001", sprintf("%.3f", P_Value))
    )

  # Format final table
  final_table <- data.frame(
    Characteristics = character(),
    `Concurrent psychotropic OR (95% CI)` = character(),
    `P value` = character(),
    stringsAsFactors = FALSE
  )

  for (pred in predictors) {
    levels_list <- levels(data[[pred]])
    ref <- ref_predictor[[pred]]
    
    pred_rows <- list()
    pred_rows[[1]] <- data.frame(
      Characteristics = ref,
      `Concurrent psychotropic OR (95% CI)` = "1 [Reference]",
      `P value` = "NA",
      stringsAsFactors = FALSE
    )

    for (lvl in levels_list) {
      if (lvl == ref) next
      term_name <- paste0(pred, lvl)
      con_row <- combined[combined$Outcome == outcomes[] & combined$term == term_name, ]

      pred_rows[[length(pred_rows) + 1]] <- data.frame(
        Characteristics = lvl,
        `Concurrent psychotropic OR (95% CI)` = if(nrow(con_row) > 0) con_row$OR_CI else "NA",
        `P value` = if(nrow(con_row) > 0) con_row$P_Value_Formatted else "NA",
        stringsAsFactors = FALSE
      )
    }

    pred_table <- do.call(rbind, pred_rows)
    pred_table <- rbind(
      data.frame(
        Characteristics = pred,
        `Concurrent psychotropic OR (95% CI)` = "",
        `P value` = "",
        stringsAsFactors = FALSE
      ),
      pred_table
    )
    final_table <- rbind(final_table, pred_table)
  }

  # Output
  cat("\nFormatted Regression Results (All Predictors Simultaneously)\n")
  cat("----------------------------------------------------------------------\n")
  print(final_table)
  write_xlsx(final_table, path = excel_file)
  cat("\nResults saved to", excel_file, "\n")
  
  invisible(final_table)
}


In [ ]:
convariates <- c("age", "sex", 'race_group', 'education_level', 'marital_status', 'income_level', 'log_exposure_std')

predictors <- c("CCI_group", "ERV_count_group", "IV_count_group")

outcomes <- c("concurrent_polypharmacy")

#Specify reference levels for predictors (match the table)
ref_predictor <- list(
  CCI_group = "0",
  ERV_count_group = "0",
  IV_count_group = "0"
)

#Specify the order of levels for each predictor (match the table)
level_order <- list(
  CCI_group = c( "0", "1-2","3-4", ">=5"),
  ERV_count_group = c("0", "1-5", "6-10", "11-20", ">=21"),
  IV_count_group = c("0", "1-5", "6-10", "11-20", ">=21")  
)
output_f <- "/working_directory/Table_6_adjusted_associations_of_polypharmacy_with_cci_ERIV.xlsx"
# Run analysis and save to Excel
poisson_regression_adjusted(all_polypharmacy_outcome_data, predictors, convariates, outcomes, 
                                           ref_predictor = ref_predictor, 
                                           level_order = level_order,                          
                                           excel_file = output_f)